# Generators and Iterators
**Module 01-02 | Core Language | Day 1**

Topics covered:
- What is an iterator and how Python's iteration protocol works
- Generator functions (`yield`)
- Generator expressions
- Memory tradeoff: list vs generator
- Real DE use cases

---
## 1. The Iteration Protocol

Any object that implements `__iter__()` and `__next__()` is an **iterator**.

Python's `for` loop is just syntactic sugar around calling these methods.

In [ ]:
# What a for loop actually does under the hood
numbers = [1, 2, 3]

# This:
for n in numbers:
    print(n)

print("--- same as ---")

# Is exactly this:
it = iter(numbers)       # calls numbers.__iter__()
while True:
    try:
        n = next(it)     # calls it.__next__()
        print(n)
    except StopIteration:
        break

In [ ]:
# Build a custom iterator from scratch
class CountUp:
    """Yields integers from start to stop (inclusive)."""
    def __init__(self, start, stop):
        self.current = start
        self.stop = stop

    def __iter__(self):
        return self  # the object itself is the iterator

    def __next__(self):
        if self.current > self.stop:
            raise StopIteration
        value = self.current
        self.current += 1
        return value

for n in CountUp(1, 5):
    print(n)

# Key insight: __next__ produces ONE value at a time and resumes from where it left off.
# The full sequence is NEVER held in memory.

---
## 2. Generator Functions — `yield`

Writing a class with `__iter__` and `__next__` is verbose. `yield` gives you the same behaviour in a normal function.

When Python sees `yield` in a function body, the function becomes a **generator function**.
Calling it returns a **generator object** — which IS an iterator.

In [ ]:
def count_up(start, stop):
    current = start
    while current <= stop:
        yield current      # pause here, hand the value to the caller
        current += 1       # resume here on the NEXT next() call

gen = count_up(1, 5)
print(type(gen))           # <class 'generator'>

print(next(gen))           # 1
print(next(gen))           # 2
print(next(gen))           # 3

# Or just iterate:
for n in count_up(1, 5):
    print(n)

In [ ]:
# Execution is SUSPENDED at yield — local state is preserved between calls
def show_state():
    print("before first yield")
    yield 1
    print("resumed after first yield")
    yield 2
    print("resumed after second yield")

g = show_state()
print("calling next() #1")
print(next(g))
print("calling next() #2")
print(next(g))
print("calling next() #3 — StopIteration")
try:
    next(g)
except StopIteration:
    print("generator exhausted")

---
## 3. Generator Expressions

Same as list comprehensions but with `()` instead of `[]`.
They are **lazy** — values are computed on demand, not all at once.

In [ ]:
# List comprehension — builds the full list in memory immediately
squares_list = [x ** 2 for x in range(10)]
print(squares_list)        # [0, 1, 4, 9, ...] — all 10 values exist right now

# Generator expression — produces values one at a time, on demand
squares_gen = (x ** 2 for x in range(10))
print(squares_gen)         # <generator object ...> — nothing computed yet
print(next(squares_gen))   # 0   — computed now
print(next(squares_gen))   # 1   — computed now

In [ ]:
# Generator expressions compose — chaining does NOT create intermediate lists
data = range(1_000_000)

# All three steps are lazy — no intermediate collection is built
result = sum(
    x ** 2
    for x in data
    if x % 2 == 0
)
print(result)

---
## 4. Memory Tradeoff: List vs Generator

A list holds all values in RAM simultaneously.
A generator holds exactly **one value** at a time (plus a tiny frame for local state).

For small data: use whichever is clearer.
For large data (millions of rows, big files): generator is mandatory.

In [ ]:
import sys

N = 1_000_000

list_obj = [x for x in range(N)]
gen_obj  = (x for x in range(N))

print(f"List size  : {sys.getsizeof(list_obj):,} bytes")
print(f"Generator  : {sys.getsizeof(gen_obj):,} bytes")

# List:      ~8 MB
# Generator: ~200 bytes — does not grow with N

In [ ]:
# Tradeoff: generators are single-pass — you cannot rewind or index them
gen = (x for x in range(5))
print(list(gen))   # [0, 1, 2, 3, 4]
print(list(gen))   # []  — generator is exhausted

# If you need to iterate multiple times, use a list.
# If you iterate once (pipeline pattern), use a generator.

---
## 5. DE Use Cases

### 5a. Reading a large CSV file without loading it all into memory

In [ ]:
def read_csv_rows(filepath):
    """Yield one parsed row dict at a time — file never fully in RAM."""
    import csv
    with open(filepath, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            yield row

# Usage — processes one row at a time regardless of file size
# for row in read_csv_rows("huge_file.csv"):
#     process(row)

In [ ]:
# Contrast: Pandas read_csv loads everything into RAM at once
# import pandas as pd
# df = pd.read_csv("huge_file.csv")   # 5GB file → 5GB+ RAM spike

# The generator approach is the foundation of how Spark/Beam stream records.
# Every Spark RDD transformation is lazy — it's a chain of generators under the hood.

### 5b. Paginated API ingestion — yield one page at a time

In [ ]:
import requests

def fetch_pages(base_url, params=None):
    """Generator that yields one page of results at a time."""
    page = 1
    params = params or {}
    while True:
        response = requests.get(base_url, params={**params, "page": page})
        response.raise_for_status()
        data = response.json()
        if not data:          # empty page → done
            return
        yield data            # caller gets one page, we pause here
        page += 1

# Usage — memory stays flat even if the API has 10,000 pages
# for page in fetch_pages("https://api.example.com/events"):
#     for record in page:
#         write_to_parquet(record)

### 5c. Pipeline pattern — chain generators without intermediate collections

In [ ]:
# Each stage is a generator — only one record flows through at a time

def extract(filepath):
    import csv
    with open(filepath, newline="") as f:
        yield from csv.DictReader(f)

def transform(rows):
    for row in rows:
        row["amount"] = float(row["amount"])
        row["amount_usd"] = row["amount"] * 1.08
        yield row

def filter_positive(rows):
    for row in rows:
        if row["amount"] > 0:
            yield row

# Compose the pipeline — nothing runs until you consume it
# pipeline = filter_positive(transform(extract("transactions.csv")))
# for record in pipeline:
#     load(record)

# This is exactly how Apache Beam's pipeline model works conceptually.

---
## 6. Quick Reference

| | List comprehension | Generator expression | Generator function |
|---|---|---|---|
| Syntax | `[x for x in ...]` | `(x for x in ...)` | `def f(): yield x` |
| Memory | All values at once | One value at a time | One value at a time |
| Re-iterable | Yes | No | No (re-call the function) |
| Indexable | Yes | No | No |
| Use when | Small data, multiple passes | One-pass aggregation | Complex logic, I/O, pipelines |

**Rule of thumb for DE:**
- File I/O, API pagination, streaming records → generator function
- One-liner filter/map with no I/O → generator expression
- You need `df[i]` or need to loop twice → list

---
## Exercises

1. Write a generator function `read_jsonl(filepath)` that yields one parsed dict per line from an NDJSON file.
2. Write a generator `batch(iterable, size)` that yields lists of `size` items — useful for bulk-insert batching.
3. Rewrite the API pagination generator to handle a `next_cursor` field instead of a page number.
4. Use `sys.getsizeof` to confirm that chaining three generator expressions uses the same memory as one.